# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

14:50:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [2]:
schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType()),
    T.StructField("nom", T.StringType()),
    T.StructField("categorie", T.StringType()),
    T.StructField("sous_categorie", T.StringType()),
    T.StructField("prix", T.DoubleType()),
    T.StructField("poids_kg", T.DoubleType()),
])
schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType()),
    T.StructField("client_id", T.StringType()),
    T.StructField("horodatage", T.TimestampType()),
    T.StructField("evenement", T.StringType()),
    T.StructField("livreur_id", T.StringType()),
    T.StructField("detail", T.StringType()),
])
schema_communes = T.StructType([
    T.StructField("commune", T.StringType()),
    T.StructField("code_postal", T.StringType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("lat", T.DoubleType()),
])
schema_gps = T.StructType([
    T.StructField("timestamp", T.TimestampType()),
    T.StructField("livreur_id", T.StringType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("lat", T.DoubleType()),
    T.StructField("vitesse_kmh", T.DoubleType()),
    T.StructField("dans_vehicule", T.BooleanType()),
    T.StructField("charge_kg", T.DoubleType()),
])

produits = spark.read.schema(schema_produits).option("header", "true").csv(DATA + "/produits.csv")
commandes = spark.read.schema(schema_commandes).option("header", "true").csv(DATA + "/commandes.csv")
communes = spark.read.schema(schema_communes).option("header", "true").csv(DATA + "/communes.csv")
gps = spark.read.schema(schema_gps).option("header", "true").csv(DATA + "/vehicules/*/*.csv")

journal = commandes.persist()  # le journal d'evenements, reutilise partout

for nom, df in [("produits", produits), ("commandes", commandes),
                ("communes", communes), ("gps", gps)]:
    print(f"{nom} : {df.count()} lignes")

produits : 233 lignes


commandes : 846535 lignes
communes : 142 lignes


gps : 5174905 lignes


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [3]:
gps = (gps
       .withColumn("vehicule_id",
                   F.regexp_extract(F.input_file_name(), r"(V\d+)/[^/]*\.csv$", 1))
       .withColumn("jour", F.to_date("timestamp")))
gps = gps.persist() 
gps.show(5)

+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|      jour|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|2025-10-01 07:40:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:41:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:42:00|      L001|2.277084|48.936734|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:43:00|      L001|2.277083|48.936734|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:44:00|      L001|2.277084|48.936733|        0.0|        false|   2581.6|        V18|2025-10-01|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
o

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [4]:
commandes.agg(
    F.countDistinct("commande_id").alias("commandes"),
    F.countDistinct("client_id").alias("clients"),
    F.countDistinct("livreur_id").alias("livreurs"),
).show()
gps.select(F.countDistinct("vehicule_id").alias("vehicules")).show()

+---------+-------+--------+
|commandes|clients|livreurs|
+---------+-------+--------+
|   271617|   8000|      90|
+---------+-------+--------+



+---------+
|vehicules|
+---------+
|       72|
+---------+



2.2) Comptez les événements du journal par type.

In [5]:
commandes.groupBy("evenement").count().orderBy(F.desc("count")).show()

+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|   livree|268682|
|    echec| 11220|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [6]:
charges = (commandes.filter(F.col("evenement") == "chargee")
           .groupBy("commande_id").count().withColumnRenamed("count", "nb_chargee"))

(commandes.filter(F.col("evenement") == "commande").select("commande_id")
 .join(charges, "commande_id", "left")
 .fillna(0, ["nb_chargee"])
 .groupBy("nb_chargee").count().orderBy("nb_chargee")).show()

+----------+------+
|nb_chargee| count|
+----------+------+
|         0|  1637|
|         1|254229|
|         2| 14673|
|         3|  1026|
|         4|    51|
|         5|     1|
+----------+------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [7]:
w_cmd = Window.partitionBy("commande_id").orderBy("horodatage")

(commandes.withColumn("ev_prec", F.lag("evenement").over(w_cmd))
 .filter(F.col("evenement") == "chargee")
 .groupBy("ev_prec").count()
 .orderBy(F.desc("count"))).show()

+--------+------+
| ev_prec| count|
+--------+------+
|commande|269980|
|   echec|  9950|
|reportee|  6932|
+--------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [8]:
etat = (commandes.groupBy("commande_id").agg(
    F.max(F.when(F.col("evenement") == "livree", 1).otherwise(0)).alias("livree"),
    F.max(F.when(F.col("evenement") == "retour", 1).otherwise(0)).alias("retour")))

etat.agg(
    F.sum("livree").alias("livrees"),
    F.sum(F.when((F.col("livree") == 0) & (F.col("retour") == 1), 1).otherwise(0))
      .alias("abandonnees"),
    F.sum(F.when((F.col("livree") == 0) & (F.col("retour") == 0), 1).otherwise(0))
      .alias("en_cours"),
).show()

+-------+-----------+--------+
|livrees|abandonnees|en_cours|
+-------+-----------+--------+
| 268682|       1194|    1741|
+-------+-----------+--------+



## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [9]:
lignes = (commandes.filter(F.col("evenement") == "commande")
    .select("commande_id", "client_id", "horodatage",
            F.explode(F.split("detail", r"\|")).alias("item"))
    .withColumn("produit_id", F.split("item", "x").getItem(0))
    .withColumn("quantite", F.split("item", "x").getItem(1).cast("int"))
    .drop("item"))
lignes.show(5)

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0037|       5|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0063|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0019|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0067|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0069|       2|
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows



3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [10]:
lignes = (lignes.join(produits, "produit_id")
          .withColumn("montant", F.round(F.col("prix") * F.col("quantite"), 2)))
lignes = lignes.persist()

commandes = (lignes.groupBy("commande_id", "client_id", "horodatage")
    .agg(F.round(F.sum("montant"), 2).alias("montant"),
         F.round(F.sum(F.col("poids_kg") * F.col("quantite")), 1).alias("poids_kg"),
         F.count("*").alias("nb_lignes")))
commandes = commandes.persist()
commandes.show(5)

+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|         horodatage|montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|   C0000168|  CL06333|2025-01-01 13:16:07| 209.35|    61.0|       32|
|   C0000469|  CL03079|2025-01-01 22:39:48|   80.9|    14.0|       13|
|   C0000752|  CL00230|2025-01-02 16:34:53| 218.05|    46.5|       31|
|   C0000810|  CL07244|2025-01-02 18:29:22| 239.45|    78.5|       27|
|   C0000855|  CL04907|2025-01-02 19:48:36|   70.9|    15.9|       15|
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows



## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [11]:
w_vj = Window.partitionBy("vehicule_id", "jour").orderBy("timestamp")

bords = (gps
    .withColumn("rn", F.row_number().over(w_vj))
    .withColumn("rni", F.row_number().over(w_vj.orderBy(F.desc("timestamp"))))
    .filter((F.col("rn") == 1) | (F.col("rni") == 1))
    .withColumn("lon_r", F.round("lon", 2))
    .withColumn("lat_r", F.round("lat", 2)))

positions = (bords.groupBy("lon_r", "lat_r").count()
             .orderBy(F.desc("count")))
positions.show(20, truncate=False)
print("positions distinctes :", positions.count())

+-----+-----+-----+
|lon_r|lat_r|count|
+-----+-----+-----+
|2.45 |48.91|8434 |
|2.28 |48.94|6416 |
|2.35 |48.75|5934 |
+-----+-----+-----+



positions distinctes : 3


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [12]:
premiers = bords.filter(F.col("rn") == 1)

entrepots = (premiers.groupBy("lon_r", "lat_r")
    .agg(F.round(F.avg("lon"), 6).alias("lon"),
         F.round(F.avg("lat"), 6).alias("lat"))
    .withColumn("entrepot",
                F.concat(F.lit("E"), F.row_number().over(Window.orderBy("lon")).cast("string"))))
entrepots.select("entrepot", "lon", "lat").show(truncate=False)

vehicule_entrepot = (premiers.groupBy("vehicule_id", "lon_r", "lat_r").count()
    .withColumn("r", F.row_number().over(
        Window.partitionBy("vehicule_id").orderBy(F.desc("count"))))
    .filter(F.col("r") == 1)
    .join(entrepots, ["lon_r", "lat_r"])
    .select("vehicule_id", "entrepot"))

vehicule_entrepot.groupBy("entrepot").count().orderBy("entrepot").show()

14:52:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:54 WARN WindowExec: No Partition Defined for Window operat

+--------+--------+---------+
|entrepot|lon     |lat      |
+--------+--------+---------+
|E1      |2.277069|48.936713|
|E2      |2.351514|48.747047|
|E3      |2.448959|48.906702|
+--------+--------+---------+



14:52:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
14:52:59 WARN WindowExec: No Partition Defined for Window operat

+--------+-----+
|entrepot|count|
+--------+-----+
|      E1|   24|
|      E2|   24|
|      E3|   24|
+--------+-----+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [13]:
w_lj = Window.partitionBy("livreur_id", "jour").orderBy("timestamp")

gps = (gps
    .withColumn("avant", F.lag("dans_vehicule").over(w_lj))
    .withColumn("changement",
        F.when(F.col("avant") & ~F.col("dans_vehicule"), 1).otherwise(0))
    .drop("avant"))

(gps.filter((F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06")
            & (F.col("changement") == 1))
 .select("timestamp", "lon", "lat", "vitesse_kmh", "changement")
 .orderBy("timestamp")).show(30, truncate=False)

+-------------------+--------+---------+-----------+----------+
|timestamp          |lon     |lat      |vitesse_kmh|changement|
+-------------------+--------+---------+-----------+----------+
|2025-01-06 09:54:00|2.311953|48.871389|0.0        |1         |
|2025-01-06 10:00:00|2.313447|48.870848|0.0        |1         |
|2025-01-06 10:09:00|2.316012|48.872919|0.0        |1         |
|2025-01-06 10:23:00|2.325412|48.874465|0.0        |1         |
|2025-01-06 10:36:00|2.32844 |48.866695|0.0        |1         |
|2025-01-06 10:54:00|2.337183|48.866435|0.0        |1         |
|2025-01-06 11:12:00|2.317523|48.857452|0.0        |1         |
|2025-01-06 11:38:00|2.317717|48.850169|0.0        |1         |
|2025-01-06 12:02:00|2.312056|48.845401|0.0        |1         |
|2025-01-06 12:15:00|2.305215|48.854413|0.0        |1         |
|2025-01-06 13:04:00|2.298876|48.845968|0.0        |1         |
|2025-01-06 13:42:00|2.326214|48.891988|0.0        |1         |
|2025-01-06 13:53:00|2.324882|48.892465|

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [14]:
gps = gps.withColumn("numero_livraison",
    F.sum("changement").over(
        w_lj.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
gps = gps.persist()

(gps.filter((F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06")
            & ~F.col("dans_vehicule"))
 .groupBy("numero_livraison")
 .agg(F.min("timestamp").alias("premier"),
      F.max("timestamp").alias("dernier"),
      F.count("*").alias("minutes_a_pied"))
 .orderBy("numero_livraison")).show(30, truncate=False)

+----------------+-------------------+-------------------+--------------+
|numero_livraison|premier            |dernier            |minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|0               |2025-01-06 07:43:00|2025-01-06 08:26:00|44            |
|1               |2025-01-06 09:54:00|2025-01-06 09:57:00|4             |
|2               |2025-01-06 10:00:00|2025-01-06 10:06:00|7             |
|3               |2025-01-06 10:09:00|2025-01-06 10:18:00|10            |
|4               |2025-01-06 10:23:00|2025-01-06 10:26:00|4             |
|5               |2025-01-06 10:36:00|2025-01-06 10:43:00|8             |
|6               |2025-01-06 10:54:00|2025-01-06 10:57:00|4             |
|7               |2025-01-06 11:12:00|2025-01-06 11:29:00|18            |
|8               |2025-01-06 11:38:00|2025-01-06 11:54:00|17            |
|9               |2025-01-06 12:02:00|2025-01-06 12:05:00|4             |
|10              |2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [15]:
livraisons = (gps.filter(~F.col("dans_vehicule"))
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(F.first("vehicule_id").alias("vehicule_id"),
         F.min("timestamp").alias("debut"),
         F.max("timestamp").alias("fin"),
         F.count("*").alias("duree_min"),
         F.round(F.avg("lon"), 5).alias("lon"),
         F.round(F.avg("lat"), 5).alias("lat")))
livraisons = livraisons.persist()
print("livraisons :", livraisons.count())

livraisons : 273593


5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [16]:
ev_liv = (journal.filter(F.col("evenement").isin("livree", "echec"))
    .select("livreur_id", "evenement", "commande_id", "client_id",
            F.col("horodatage").alias("signature"),
            F.to_date("horodatage").alias("jour")))

l, e = livraisons.alias("l"), ev_liv.alias("e")
livraisons_ev = (l.join(e,
    (F.col("e.livreur_id") == F.col("l.livreur_id")) &
    (F.col("e.jour") == F.col("l.jour")) &
    (F.col("e.signature") >= F.col("l.debut") - F.expr("interval 1 minute")) &
    (F.col("e.signature") <= F.col("l.fin") + F.expr("interval 1 minute")),
    "left")
    .select("l.*", "e.signature", "e.evenement", "e.commande_id", "e.client_id"))
livraisons_ev = livraisons_ev.persist()
livraisons_ev.show(10, vertical=True)

-RECORD 0-------------------------------
 livreur_id       | L002                
 jour             | 2025-05-09          
 numero_livraison | 0                   
 vehicule_id      | V03                 
 debut            | 2025-05-09 07:41:00 
 fin              | 2025-05-09 08:23:00 
 duree_min        | 43                  
 lon              | 2.27706             
 lat              | 48.93673            
 signature        | NULL                
 evenement        | NULL                
 commande_id      | NULL                
 client_id        | NULL                
-RECORD 1-------------------------------
 livreur_id       | L002                
 jour             | 2025-05-09          
 numero_livraison | 1                   
 vehicule_id      | V03                 
 debut            | 2025-05-09 10:37:00 
 fin              | 2025-05-09 10:41:00 
 duree_min        | 5                   
 lon              | 2.31607             
 lat              | 48.84265            
 signature      

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [17]:
rattaches = (livraisons_ev.filter(F.col("evenement").isNotNull())
    .select(F.col("signature").alias("s"), F.col("commande_id").alias("c")))

non_rattaches = ev_liv.join(
    rattaches,
    (ev_liv.signature == rattaches.s) & (ev_liv.commande_id == rattaches.c),
    "left_anti")
print("evenements livree/echec non rattaches :", non_rattaches.count())

evenements livree/echec non rattaches : 0


5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

In [18]:
nb_ev = (livraisons_ev.filter(F.col("evenement").isNotNull())
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(F.count("*").alias("nb_ev")))

dmin = (livraisons.crossJoin(
        entrepots.select(F.col("lon").alias("e_lon"), F.col("lat").alias("e_lat")))
    .withColumn("d", haversine("lon", "lat", "e_lon", "e_lat"))
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(F.min("d").alias("d_entrepot")))

livraisons_ev = (livraisons_ev
    .join(nb_ev, ["livreur_id", "jour", "numero_livraison"], "left")
    .join(dmin, ["livreur_id", "jour", "numero_livraison"], "left")
    .fillna(0, ["nb_ev"])
    .withColumn("type",
        F.when(F.col("nb_ev") > 0, "livraison")
         .when(F.col("d_entrepot") < 0.3, "entrepot")
         .otherwise("pause")))
livraisons_ev = livraisons_ev.persist()

(livraisons_ev
 .select("livreur_id", "jour", "numero_livraison", "type", "duree_min", "debut")
 .distinct()
 .groupBy("type")
 .agg(F.count("*").alias("nb"),
      F.round(F.avg("duree_min"), 1).alias("duree_moy_min"),
      F.round(F.avg(F.hour("debut") + F.minute("debut") / 60.0), 2)
        .alias("heure_moy_debut"))
 .orderBy("type")).show()

+---------+------+-------------+---------------+
|     type|    nb|duree_moy_min|heure_moy_debut|
+---------+------+-------------+---------------+
| entrepot| 22368|         27.2|          11.68|
|livraison|246258|          8.3|          12.19|
|    pause|  4967|         37.5|          12.35|
+---------+------+-------------+---------------+



5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

In [19]:
liv_uniq = livraisons_ev.select(
    "livreur_id", "jour", "numero_livraison", "nb_ev").distinct()
liv_uniq.groupBy("nb_ev").count().orderBy("nb_ev").show()

multi = (livraisons_ev.filter(F.col("evenement").isNotNull())
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(F.countDistinct("client_id").alias("nb_clients"))
    .filter(F.col("nb_clients") >= 2))
print("livraisons concernant au moins 2 clients :", multi.count())

+-----+------+
|nb_ev| count|
+-----+------+
|    0| 27335|
|    1|217538|
|    2| 24873|
|    3|  3053|
|    4|   592|
|    5|   155|
|    6|    29|
|    7|     8|
|    8|     5|
|    9|     4|
|   10|     1|
+-----+------+



livraisons concernant au moins 2 clients : 12536


5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

In [20]:
w_ev = (Window.partitionBy("livreur_id", "jour", "numero_livraison")
        .orderBy("signature"))

livraisons_ev = (livraisons_ev
    .withColumn("sig_prec", F.lag("signature").over(w_ev))
    .withColumn("duree_ev_min",
        F.when(F.col("signature").isNull(), F.lit(None))
         .otherwise(F.round(
             (F.col("signature").cast("long")
              - F.coalesce(F.col("sig_prec"), F.col("debut")).cast("long")) / 60.0, 1)))
    .drop("sig_prec"))

(livraisons_ev.filter(F.col("signature").isNotNull())
 .groupBy("evenement")
 .agg(F.round(F.avg("duree_ev_min"), 1).alias("duree_moy_min"))).show()

+---------+-------------+
|evenement|duree_moy_min|
+---------+-------------+
|    echec|          2.0|
|   livree|          4.2|
+---------+-------------+



5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

In [21]:
signes = (livraisons_ev.filter(F.col("signature").isNotNull())
    .groupBy("livreur_id")
    .agg(F.sum(F.when(F.col("evenement") == "livree", 1).otherwise(0)).alias("signees"),
         F.sum(F.when(F.col("evenement") == "echec", 1).otherwise(0)).alias("echecs")))
jours_trav = gps.groupBy("livreur_id").agg(F.countDistinct("jour").alias("jours"))

(signes.join(jours_trav, "livreur_id")
 .withColumn("signees_par_jour", F.round(F.col("signees") / F.col("jours"), 1))
 .orderBy(F.desc("signees_par_jour"))).show(10)

+----------+-------+------+-----+----------------+
|livreur_id|signees|echecs|jours|signees_par_jour|
+----------+-------+------+-----+----------------+
|      L076|    468|    24|   15|            31.2|
|      L089|    804|    42|   26|            30.9|
|      L070|    543|    28|   18|            30.2|
|      L086|    713|    26|   24|            29.7|
|      L081|    615|    18|   21|            29.3|
|      L072|    490|    21|   17|            28.8|
|      L079|    459|    16|   16|            28.7|
|      L080|    457|    18|   16|            28.6|
|      L085|    715|    36|   25|            28.6|
|      L082|    451|    19|   16|            28.2|
+----------+-------+------+-----+----------------+
only showing top 10 rows



## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*